# 🚀 Kaggle Worker — vLLM (Qwen 2.5 14B AWQ) + 2x T4 GPU (Tensor Parallelism = 2)

**Ưu điểm vượt trội so với Ollama:**
1. **Tensor Parallelism (TP=2)**: Xẻ đôi ma trận tính toán sang CẢ 2 GPU T4 $\rightarrow$ Cả 2 GPU chạy 100% công suất.
2. **Tốc độ cực nhanh**: 35 - 50 tokens/giây (gấp 3-4 lần Ollama 14B).
3. **Continuous Batching**: Xử lý 4-8 request dịch đồng thời mà không bị nghẽn hay rớt batch.
4. **Chuẩn OpenAI API**: Tương thích hoàn hảo với `/v1/chat/completions`.

In [ ]:
!pip install huggingface_hub
from huggingface_hub import snapshot_download

# Tải toàn bộ file model về thư mục chỉ định
snapshot_download(
    repo_id="Qwen/Qwen2.5-14B-Instruct-AWQ",
    local_dir="/kaggle/working/Qwen2.5-14B-AWQ"
)

In [ ]:
import os
os.makedirs('/kaggle/working/wheels', exist_ok=True)

print("Đang tải các file cài đặt (.whl) về máy...")
!pip download vllm pyngrok httpx -d /kaggle/working/wheels

print("Hoàn tất tải file!")

In [ ]:
# ─── [1/4] Cài đặt vLLM & Dependencies (OFFLINE) ────────────────────────────
print('🚀 [1/4] Đang cài đặt vLLM & pyngrok từ ổ cứng (siêu tốc)...')

# Thay đường dẫn bên dưới bằng đường dẫn thực tế của bạn
WHEELS_DIR = '/kaggle/input/notebooks/baonguyenvan/download-qwen-awqandvllm-offline-packages/wheels'

# --no-index: Chặn pip kết nối internet
# --find-links: Yêu cầu pip tìm file cài đặt trong thư mục nội bộ
!pip install --quiet --no-index --find-links={WHEELS_DIR} vllm pyngrok httpx

print('✅ Cài đặt thành công vLLM!')

In [ ]:
# ─── [2/4] Khởi động vLLM Server (OFFLINE MODEL) ───────────────────────────
import subprocess, time, httpx

# Chú ý: Sửa 'download-qwen-awq' thành tên notebook tải model của bạn nếu đặt tên khác
LOCAL_MODEL_PATH = '/kaggle/input/notebooks/baonguyenvan/download-qwen-awqandvllm-offline-packages/Qwen2.5-14B-AWQ'
SERVED_MODEL_NAME = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
PORT = 8000

print(f'⚡ [2/4] Khởi động vLLM Server từ ổ cứng tại {LOCAL_MODEL_PATH}...')

cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', LOCAL_MODEL_PATH,
    '--served-model-name', SERVED_MODEL_NAME,
    '--tensor-parallel-size', '2',
    '--gpu-memory-utilization', '0.90',
    '--max-model-len', '4096',
    '--dtype', 'half',
    '--port', str(PORT),
    '--host', '0.0.0.0'
]

vllm_process = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('⏳ Chờ vLLM nạp model vào GPU (rất nhanh do load từ ổ cứng)...', end='')
for i in range(60):  # Tối đa 5 phút
    try:
        r = httpx.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5.0)
        if r.status_code == 200:
            print(f'\n✅ vLLM sẵn sàng sau {(i+1)*5}s!')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(5)
else:
    raise RuntimeError('❌ vLLM không khởi động được. Hãy chạy lệnh !ls /kaggle/input/ ở một cell mới để kiểm tra lại đường dẫn thư mục xem đã chính xác chưa.')

In [ ]:
# ─── [3/4] Test Inference nhanh localhost ───────────────────────────────────
import httpx

print('🧪 [3/4] Test inference qua vLLM OpenAI API...')
payload = {
    'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ',
    'messages': [
        {'role': 'user', 'content': 'Dịch sang tiếng Việt trong 1 câu ngắn: Artificial intelligence is transforming the world.'}
    ],
    'temperature': 0.2
}

r = httpx.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=60.0)
if r.status_code == 200:
    ans = r.json()['choices'][0]['message']['content'].strip()
    print(f'✅ Inference OK! Kết quả: "{ans}"')
else:
    raise RuntimeError(f'❌ Inference FAILED: {r.status_code} — {r.text[:200]}')

In [ ]:
# ─── [4/4] Mở Ngrok Static Tunnel & Self-Test End-to-End ─────────────────────
import time, httpx
from pyngrok import ngrok

NGROK_AUTHTOKEN = '3Hl420hPJiQrcgDIWpked30M1Ca_7rfpvf1wR4MzLUK4pmMPw'
STATIC_DOMAIN   = 'bondless-immerse-paternal.ngrok-free.dev'

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()
time.sleep(1)

print(f'🔗 [4/4] Đang mở Ngrok tunnel tới port 8000 (vLLM) → {STATIC_DOMAIN} ...')
tunnel = ngrok.connect(8000, domain=STATIC_DOMAIN)
PUBLIC_URL = tunnel.public_url.replace('http://', 'https://')

print(f'\n{"="*60}')
print(f'🎉 KẾT NỐI vLLM THÀNH CÔNG!')
print(f'📌 PUBLIC URL: {PUBLIC_URL}')
print(f'{"="*60}')

# Self-test qua Public URL
print('\n🧪 Self-test end-to-end qua public URL...')
BYPASS_HEADERS = {
    'ngrok-skip-browser-warning': 'true',
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/125.0.0.0 Safari/537.36',
    'Accept': 'application/json',
}

try:
    r_health = httpx.get(f'{PUBLIC_URL}/v1/models', headers=BYPASS_HEADERS, timeout=20.0)
    assert r_health.status_code == 200, f'HTTP {r_health.status_code}'
    print('   ✅ Health check PASSED!')
except Exception as e:
    print(f'   ❌ Health check FAILED: {e}')
    raise

print(f"""
{"="*60}
🚀 TẤT CẢ TEST ĐÃ PASS!

👉 BƯỚC TIẾP THEO TRÊN LOCAL (file .env):
   OLLAMA_HOST="{PUBLIC_URL}"
   OLLAMA_MODEL="Qwen/Qwen2.5-14B-Instruct-AWQ"
   USE_VLLM="true"

   Sau đó restart docker worker:
   docker compose restart doc-translation-worker
{"="*60}
""")

# Heartbeat loop
count = 0
while True:
    time.sleep(30)
    count += 30
    try:
        r = httpx.get('http://127.0.0.1:8000/v1/models', timeout=5.0)
        print(f'[Heartbeat {count}s] ✅ vLLM 2x T4 GPU đang chạy ổn định | Ngrok: {PUBLIC_URL}')
    except Exception as err:
        print(f'[Heartbeat {count}s] ⚠️ Cảnh báo: {err}')